# Afternoon class 30/08 — Worksheet 11 SOLUTIONS: reading CSV   (L03)

Every cell below was executed in the lab image (pandas 3.0.5) against the real
files in `data/`, and the quoted output is what it actually printed.

Questions 4 and 6 are the ones to re-read. Both are cases where a wrong answer
arrives as a perfectly ordinary DataFrame.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 11 — Reading CSV and TSV. Run this once.
import pandas as pd

# All of these describe the SAME 300 orders, stored four different ways.
#   data/sales.csv         clean, comma-separated
#   data/sales.tsv         tab-separated
#   data/sales_report.csv  three metadata lines, a blank line, then the header
#   data/sales_messy.csv   '?' and 'Missing' used as placeholders

sales = pd.read_csv("data/sales.csv")
print("sales.csv ->", sales.shape)
print(list(sales.columns))

PART A — the default read, and what to check

### Question 1

`(300, 10)`, no missing values. -> `OrderID/Quantity int64`, `Sales/Discount/Profit float64`, the rest `str`.

The default read did the right thing on a clean file, which is the deck's
point: do not add parameters until something is wrong.

The three checks worth making every single time are here. **Shape** — is it
the number of rows you expected? **dtypes** — did the numeric columns come
in as numbers? **`isna().sum().sum()`** — is anything missing? Ten seconds,
and they catch most load-time disasters before they become analysis.

Note `OrderDate` is `str`. It looks like a date and Pandas has not treated
it as one; Q8 and Q9 are about what that costs.

In [ ]:
print(sales.head(3)[["OrderID", "OrderDate", "Region", "Sales"]])
print()
print("shape:", sales.shape)
print()
print(sales.dtypes)
print()
print("missing values:", sales.isna().sum().sum())

### Question 2

`sep="\t"` -> `(300, 10)` and `.equals(sales)` is `True`.

Byte-for-byte the same 300 orders. CSV and TSV are the same format with a
different separator, and `sep=` is the whole difference.

In [ ]:
tsv = pd.read_csv("data/sales.tsv", sep="\t")
print("shape:", tsv.shape)
print("identical to sales.csv:", tsv.equals(sales))

### Question 3

The real TSV -> **raises** `ParserError: Expected 2 fields in line 18, saw 3`. The comma-free TSV -> **`(2, 1)`, no error**, with one column literally named `'Region\tSales'`.

Same mistake, two completely different outcomes.

The real file raised because its `Product` names contain commas — 301 of
the 1,237 products do. Reading it comma-separated, the header line has no
commas (1 field) and line 18 has two (3 fields), so the row lengths
disagree and the parser gives up. The error is confusing but it is an
error, and it stops you.

The comma-free file is the dangerous one. Every line has exactly zero
commas, so every row has consistently one field, and Pandas hands you a
perfectly well-formed `(2, 1)` DataFrame whose single column is named
`Region\tSales` and whose values are `Ontario\t151.35`. `.shape` works.
`.head()` works. Nothing raises. You have one column of garbage and a
program that runs.

So whether a wrong `sep` is caught depends on whether your data happens to
contain the other delimiter. That is not a property you control.

In [ ]:
try:
    wrong = pd.read_csv("data/sales.tsv")
    print("shape:", wrong.shape)
except Exception as exc:
    print("%s: %s" % (type(exc).__name__, exc))

print()
# Same error, on data that contains no commas anywhere.
with open("/tmp/clean.tsv", "w") as fh:
    fh.write("Region\tSales\n")
    fh.write("Ontario\t151.35\n")
    fh.write("West\t51.56\n")

quiet = pd.read_csv("/tmp/clean.tsv")
print("no-comma TSV read as CSV -> shape", quiet.shape)
print("columns:", list(quiet.columns))
print(quiet)

PART B — the metadata preamble

### Question 4

`skiprows=2` -> `(301, 1)`, first column `'Currency: CAD'`. `=3` -> `(300, 10)`. `=4` -> `(300, 10)`. `=5` -> `(299, 10)`, first column `'8710'`.

Four values, three wrong answers, **zero exceptions**.

- **2** treats `Currency: CAD` as the header. One column, 301 rows, and
  every line of real data crammed into it.
- **3** and **4** both work — see Q5.
- **5** eats the header row. The first order's values become the column
  names, so you have a column called `8710` and 299 rows instead of 300.
  One order silently became your schema.

The `skiprows=5` case is the one to fear. `(299, 10)` looks entirely
plausible. If you did not already know the file has 300 rows, nothing in
that output would tell you a record is missing.

This is why the deck's advice to check `.shape` and `.head()` after every
load is not a formality.

In [ ]:
for n in (2, 3, 4, 5):
    d = pd.read_csv("data/sales_report.csv", skiprows=n)
    print("skiprows=%d -> shape %-11s first column %r"
          % (n, str(d.shape), list(d.columns)[0]))

### Question 5

Lines 1-3 are the metadata, **line 4 is `'\n'` — blank** — and line 5 is the header.

`read_csv` defaults to `skip_blank_lines=True`, so the blank line is
discarded regardless of whether you skipped it. `skiprows=3` skips the
three metadata lines and then the parser drops the blank one itself;
`skiprows=4` skips the blank one explicitly. Both arrive at line 5.

That forgiveness is pleasant and it is why nobody learns the real
structure of their file. The value that works is a range, not a number, and
the range has a hard edge at 5 where you start eating data.

Reading the first few raw lines with plain `open()` takes one cell and
removes all the guessing. Do it once per new file.

In [ ]:
with open("data/sales_report.csv") as fh:
    for i, line in enumerate(fh):
        if i >= 5:
            break
        print("line %d: %r" % (i + 1, line))

# Line 4 is blank. read_csv's skip_blank_lines defaults to True, so the
# blank line is discarded whether you skipped it or not -- which is why
# skiprows=3 and skiprows=4 both land on the header.

PART C — placeholders that are not missing values

### Question 6

`Discount` -> **`str`**, `Region` -> `str`, `Profit` -> `float64` with **24** missing. -> `isna().sum()` reports missing only in `Profit`; `Discount` unique values include `'?'`.

This is the most dangerous result in the sheet, and it is dangerous
because `isna().sum()` — the check recommended everywhere, including Q1 —
reports almost nothing wrong.

The `?` characters are not missing values as far as Pandas is concerned.
They are ordinary text. And one piece of text in a column of numbers forces
the **entire column** to `str`. So `Discount` has 43 corrupted entries and
**zero** reported `NaN`s.

`Profit` is different: its placeholder was a genuinely empty field, which
`read_csv` does recognise, so those 24 became real `NaN` and the column
stayed `float64`.

Two columns damaged the same way by the same file, and only one of them is
visible to the standard check. The tell is the **dtype**: a column you know
is numeric reading as `str` means something non-numeric is hiding in it.

In [ ]:
messy = pd.read_csv("data/sales_messy.csv")
print("Discount dtype:", messy["Discount"].dtype)
print("Region dtype:  ", messy["Region"].dtype)
print("Profit dtype:  ", messy["Profit"].dtype)
print()
print("isna().sum():")
print(messy.isna().sum())
print()
print("first few Discount values:", messy["Discount"].unique()[:6])

### Question 7

`.mean()` on the text column -> **`TypeError: Cannot perform reduction 'mean' with string dtype`**. With `na_values=["?", "Missing"]` -> `Discount` becomes `float64`; missing counts `Region 28`, `Discount 43`, `Profit 24`; mean `0.047976653696498055`.

The `TypeError` is the good outcome — it is what eventually forces you to
look. Q6 was the bad outcome, where the damage sat there silently.

With `na_values` the true picture appears: 43 discounts and 28 regions were
never recorded. Those 71 holes existed all along; the file just spelled
them in a way Pandas had no reason to recognise.

And note the mean is now computed over 257 values, not 300. `.mean()`
skips `NaN` silently, so the number is real but it is the average of the
discounts that were recorded — which is a different quantity from the
average discount, if the missing ones are missing for a reason.

`na_values` is the parameter you cannot skip on real exports. Every source
system has its own spelling of nothing: `?`, `N/A`, `-`, `NULL`, `9999`.

In [ ]:
messy = pd.read_csv("data/sales_messy.csv")
try:
    print("mean without na_values:", messy["Discount"].mean())
except Exception as exc:
    print("mean without na_values -> %s: %s" % (type(exc).__name__, exc))

print()
fixed = pd.read_csv("data/sales_messy.csv", na_values=["?", "Missing"])
print("Discount dtype now:", fixed["Discount"].dtype)
print()
print(fixed.isna().sum())
print()
print("mean with na_values:", fixed["Discount"].mean())

### Question 8

`usecols` -> `(300, 2)`. `OrderDate` `str` -> **`datetime64[us]`** with `parse_dates`. -> earliest `2009-01-04`, latest `2012-12-08`.

`usecols` reads only the columns you name — less memory and a faster read,
and on a 50-column file it is the difference between a workable frame and
an unreadable one.

`parse_dates` converts the text into real timestamps, which is what makes
`.min()`, `.max()`, `.dt.quarter` and date arithmetic mean anything.

The dtype is `datetime64[us]` — **microseconds**. Older Pandas always used
nanoseconds (`datetime64[ns]`), which could only span 1678-2262. Pandas 3
picks a resolution to fit the data, so older tutorials asserting `[ns]`
will not match what you see.

In [ ]:
narrow = pd.read_csv("data/sales.csv", usecols=["OrderDate", "Sales"])
print("usecols shape:", narrow.shape)
print()
print("OrderDate dtype without parse_dates:", sales["OrderDate"].dtype)
dated = pd.read_csv("data/sales.csv", parse_dates=["OrderDate"])
print("OrderDate dtype with parse_dates:   ", dated["OrderDate"].dtype)
print()
print("now date arithmetic works:")
print("earliest:", dated["OrderDate"].min())
print("latest:  ", dated["OrderDate"].max())

### Question 9

For these ISO dates, text and parsed give the same `max`. -> For `DD/MM/YYYY`, text max is **`21/01/2009`** while the true max is **`2011-12-03`**.

`YYYY-MM-DD` is the one format where alphabetical order and chronological
order coincide — the most significant component is leftmost and every field
is zero-padded. So `sales["OrderDate"].max()` gave the right answer despite
being a string comparison.

It gave the right answer for the wrong reason, and the reason stops holding
the moment the format changes. In `DD/MM/YYYY`, text comparison sorts on
the **day** first: `21/01/2009` beats `03/12/2011` because `2` beats `0`.
The 'latest' date is nearly three years early, and no error is raised.

This is why ISO-8601 is worth insisting on in any file you control, and
why `parse_dates` is worth using even when the text seems to sort fine.

In [ ]:
dated = pd.read_csv("data/sales.csv", parse_dates=["OrderDate"])
print("as text: ", sales["OrderDate"].max())
print("as dates:", dated["OrderDate"].max())
print()
demo = pd.Series(["03/12/2011", "21/01/2009"])
print("the two dates:      ", list(demo))
print("max as TEXT:        ", demo.max(), "<- the EARLIER date wins")
print("max as DATES:       ", pd.to_datetime(demo, format="%d/%m/%Y").max())

### Question 10

`pd.read_csv("data/sales_2027.csv")` -> **raises** `FileNotFoundError: [Errno 2] No such file or directory: 'data/sales_2027.csv'`.

The path is relative, so it is resolved against wherever the notebook is
running — not where the notebook file lives. If this raises for a file you
can see in the browser, check `os.getcwd()` before you check your spelling.

It is worth appreciating that this is the *loudest* failure in the sheet
and also the least harmful. A missing file stops you instantly. A wrong
`sep`, a wrong `skiprows` or an unrecognised `?` all hand you a DataFrame
and let you carry on.

In [ ]:
print(pd.read_csv("data/sales_2027.csv"))